# 91 · Prueba de v15: triaje + rasgos + recuadro en una consulta, 5 registros por consulta

**v15** (`prompts/assist_v15.txt`) hace todo en una consulta. Para cada registro y cada foto: contenido → huesos visibles → si hay 1: recuadro, lado, erosión y calidad (cubierta/foco, como v14), mirando dentro del recuadro. Al final, el resumen del registro (como el triaje). Varios registros por consulta, cada uno marcado "NEW RECORD, evaluate from scratch", en orden barajado con semilla fija.

**Pregunta:** ¿v15 propone igual de bien que v14 + triaje, y meter 5 registros en una consulta empeora algo (arrastre entre registros)?

**Cómo:** en el piloto A1, ya anotado a ciegas:
- **v15 de a 5** (`_p5`): ~14 consultas para 66 registros.
- **v15 de a 1** (`_p1`): ~66 consultas; memoria limpia por registro, la referencia.
- Las dos contra **v14 + triaje_v2** (ya corridos) y contra la anotación manual, en las mismas fotos.
- Recuadros: a la vista en A1 e IoU contra los recuadros a mano de A2 (~10 consultas).

Total: ~90 consultas de Gemini (de las 500 del día). Los resultados van a versiones con sufijo (`photo_traits_v15_p5`, etc.): no tocan nada de lo anterior.

In [ ]:
# --- Preparacion (correr siempre primero) ---
# Monta Drive, clona o actualiza el repo y carga el codigo recien bajado.
from google.colab import drive, output, userdata
drive.mount('/content/drive')
output.enable_custom_widget_manager()          # interfaces (ipywidgets) en Colab

import os, sys, importlib
if not os.path.exists('/content/sepia-inaturalist'):     # sesion nueva: clonar el repo
    token = userdata.get('GITHUB_TOKEN')
    !git clone -q https://{token}@github.com/hugozeballos/sepia-inaturalist.git /content/sepia-inaturalist
%cd /content/sepia-inaturalist
!git pull -q
!git log -1 --oneline
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')   # nunca en el codigo
os.environ['PYTHONPATH'] = '/content/sepia-inaturalist/src'     # para los !python scripts/...

sys.path.insert(0, '/content/sepia-inaturalist/src')
for m in [m for m in sys.modules if m == 'sepia' or m.startswith('sepia.')]:
    del sys.modules[m]                          # descartar la version vieja en memoria
sys.path_importer_cache.pop('/content/sepia-inaturalist/src', None)
importlib.invalidate_caches()
import sepia

# Los 4 prompts que se usan desde el 4 nov (detalle en docs/prompt_versions.md)
VERSIONS = ['photo_traits_v14', 'photo_traits_v13', 'photo_traits_v12', 'photo_traits_v8']
ANNOTATOR = 'HZ'
BONEBOX = 'bonebox_v1'   # recuadro del hueso (solo fotos con 1 hueso segun v14)

In [ ]:
from pathlib import Path
import pandas as pd
from sepia.annotate import load_events

cfg = sepia.load_config('configs/pilot_a1.yaml')
ind = load_events(cfg)['individual']
v14 = sepia.load_photo_traits(cfg, 'photo_traits_v14')
IDS = sorted(set(ind['observation_id'].astype(int)) & set(v14['observation_id'].astype(int)))
IDS_STR = ','.join(map(str, IDS))
obs, pho = sepia.load_tables(cfg)
print(f"Registros de prueba (anotados y con v14): {len(IDS)} · fotos: {pho['observation_id'].isin(IDS).sum()}")

## 1. Una consulta de prueba (5 registros)

Si responde bien, la línea dice `ok 5`. Si algún registro vuelve incompleto, se repite solo.

In [ ]:
!python scripts/assist.py --config configs/pilot_a1.yaml --pack-size 5 --suffix _p5 --limit 1 --ids {IDS_STR}

## 2. v15 de a 5 registros (el resto, ~13 consultas)

In [ ]:
!python scripts/assist.py --config configs/pilot_a1.yaml --pack-size 5 --suffix _p5 --ids {IDS_STR}

## 3. v15 de a 1 registro (~66 consultas, la referencia sin arrastre)

Se puede saltar si se quiere ahorrar cuota: la comparación principal es v15 de a 5 contra v14 y contra la persona.

In [ ]:
!python scripts/assist.py --config configs/pilot_a1.yaml --pack-size 1 --suffix _p1 --ids {IDS_STR}

## 4. Resultados: rasgos por foto contra la anotación manual (mismas fotos)

- **count**: huesos visibles por foto; **side / erosion / photo_quality**: fotos con 1 hueso.
- *side when decided*: acierto del lado cuando el modelo no dice `unclear`.
- Con ~70 fotos, diferencias de ±5-10 % no son significativas.

In [ ]:
VERS = {'v14 (1 registro, sin triaje)': 'photo_traits_v14',
        'v15 de a 1': 'photo_traits_v15_p1',
        'v15 de a 5': 'photo_traits_v15_p5'}
loaded = {k: sepia.load_photo_traits(cfg, v) for k, v in VERS.items()}
loaded = {k: d for k, d in loaded.items() if not d.empty}
common = set.intersection(*[set(d['photo_id'].astype(int)) for d in loaded.values()])
print(f"Fotos con respuesta de todas las versiones: {len(common)}")
KEYS = ['count_agreement', 'side_agreement', 'side_agreement_when_decided', 'side_unclear_model',
        'erosion_agreement', 'photo_quality_agreement', 'count_n', 'side_n']
res = {}
for k in loaded:
    m = sepia.evaluate_photo_traits(cfg, VERS[k], photo_ids=common, save=False, quiet=True).attrs['metrics']
    res[k] = {key: m.get(key) for key in KEYS}
display(pd.DataFrame(res).style.format(lambda v: f"{v:.0%}" if isinstance(v, float) else v))

In [ ]:
# Arrastre entre registros: ¿de a 5 responde distinto que de a 1, foto por foto?
if 'v15 de a 1' in loaded and 'v15 de a 5' in loaded:
    a = loaded['v15 de a 1'].set_index('photo_id'); b = loaded['v15 de a 5'].set_index('photo_id')
    both = sorted(set(a.index) & set(b.index))
    for c in ['n_cuttlebones_visible', 'side', 'erosion', 'photo_quality']:
        same = (a.loc[both, c].astype(str) == b.loc[both, c].astype(str)).mean()
        print(f"{c:22s} igual de a 1 y de a 5: {same:.0%}  ({len(both)} fotos)")
    print("(referencia: dos corridas iguales del mismo prompt tampoco coinciden 100 %)")

## 5. Triaje: estado y número de huesos por registro

`n_manual` = letras distintas anotadas (huesos del registro según la persona).

In [ ]:
t = cfg.triage
letters = ind[ind['observation_id'].isin(IDS)].groupby('observation_id')['individual'].nunique()
rows = {}
for name, ver in [('triage_v2', 'triage_v2'), ('v15 de a 1', 'triage_v15_p1'), ('v15 de a 5', 'triage_v15_p5')]:
    tri = sepia.triage.load_triage(cfg, t['provider'], t['model'], ver)
    if tri.empty:
        continue
    tri = tri[tri['observation_id'].isin(IDS)].set_index('observation_id')
    n = tri['n_cuttlebones'].reindex(letters.index)
    rows[name] = {'registros': int(n.notna().sum()),
                  'huesos = persona': float((n == letters).mean()),
                  'estado cuttlebone': float((tri['record_state'] == 'cuttlebone').mean())}
display(pd.DataFrame(rows).style.format(lambda v: f"{v:.0%}" if isinstance(v, float) else v))

## 6. Recuadros

**A1** (no tiene recuadros a mano): se miran. **A2** (`a2_test`, con recuadros dibujados): IoU de v15 de a 5 (0 = no se tocan, 1 = idénticos; ≥ 0,5 cuenta como acierto). En A2 se corre v15 solo en los registros con recuadros a mano (~10 consultas).

In [ ]:
from PIL import Image, ImageDraw
boxes = sepia.bonebox.model_boxes(cfg, 'bonebox_v15_p5')
d1024 = sepia.paths(cfg)['base'] / 'photos' / '1024px'
thumbs = []
for pid, b in list(boxes.items())[:30]:
    f = pho.loc[pho['photo_id'] == pid, 'file_name'].iloc[0]
    im = Image.open(d1024 / (f.rsplit('.', 1)[0] + '.jpg')).convert('RGB')
    W, H = im.size
    ImageDraw.Draw(im).rectangle([b['x0'] * W, b['y0'] * H, b['x1'] * W, b['y1'] * H], outline='red', width=6)
    im.thumbnail((300, 300)); thumbs.append(im)
if thumbs:
    cols = 5; grid = Image.new('RGB', (cols * 305, -(-len(thumbs) // cols) * 305), 'white')
    for i, im in enumerate(thumbs):
        grid.paste(im, ((i % cols) * 305, (i // cols) * 305))
    display(grid)
print(f"{len(boxes)} fotos con recuadro de v15 (se muestran hasta 30)")

In [ ]:
cfg_a2 = sepia.load_config('configs/a2_test.yaml')
ind2 = load_events(cfg_a2)['individual']
one = ind2.groupby('photo_id').filter(lambda g: len(g) == 1)
with_box = one[one['crop_box'].apply(lambda b: isinstance(b, dict))] if 'crop_box' in one else one.iloc[0:0]
ids_a2 = sorted(set(with_box['observation_id'].astype(int)))
print(f"A2: {len(with_box)} fotos con recuadro a mano en {len(ids_a2)} registros")
if ids_a2:
    sepia.run_assist(cfg_a2, pack_size=5, suffix='_p5', ids=ids_a2)
    cmp = sepia.bonebox.compare_with_manual(cfg_a2, 'bonebox_v15_p5')
    v = cmp['iou_yx'].dropna()
    print(f"IoU mediana {v.median():.2f} · aciertos (IoU >= 0,5) {(v >= 0.5).mean():.0%} ({len(v)} fotos)")
    display(cmp)

## 7. Consultas y tokens

In [ ]:
from sepia.io import model_output_path
for c, suf in [(cfg, '_p1'), (cfg, '_p5'), (cfg_a2, '_p5')]:
    f = model_output_path(c, 'assist', c.photo_traits['provider'], c.photo_traits['model'], 'assist_v15' + suf)
    if not f.exists():
        continue
    r = sepia.read_jsonl(f)
    print(f"{Path(c.base_dir).name} {suf}: {len(r)} consultas · errores {int(r['error'].notna().sum())} · "
          f"tokens {int(r['input_tokens'].sum())} entrada / {int(r['output_tokens'].sum())} salida · "
          f"segundos por consulta (mediana) {r['elapsed_s'].median()}")

## 8. Cómo decidir

- Si **v15 de a 5** queda parecido a **v14** contra la persona (conteo, lado, erosión, calidad) y el triaje no empeora: usar v15 de a 5 en los sectores. Son ~1/5 de consulta por registro.
- Si **de a 5** empeora claro frente a **de a 1**: usar v15 de a 1 (1 consulta por registro, igual mucho menos que hoy).
- Si v15 empeora frente a v14 incluso de a 1: el prompt unificado no sirve; volver a v14 solo + triaje.

**Para usar v15 en una corrida** (sin sufijo): `!python scripts/assist.py --config configs/sector_b.yaml` y, en la anotación (notebook 02), `VERSIONS = ['photo_traits_v15']`, `BONEBOX = 'bonebox_v15'` y `cfg.triage = {**cfg.triage, 'prompt_version': 'triage_v15'}`.